# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#!pip install moviepy==1.0.3
#!pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_11972\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [9]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 3 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (27 articulos)...
  Procesando: Estrasburgo rechaza paralizar la eutanasia de Noelia...
    -> Texto completo (3372 chars)
  Procesando: Las rebajas fiscales del Gobierno ante la guerra de Irá...
    -> Texto completo (1457 chars)
  Procesando: La receta electrónica llegará a partir del 7 de abril a...
    -> Texto completo (4477 chars)
  Procesando: Trump busca a la desesperada una salida a la guerra en ...
    -> Texto completo (8851 chars)
  Procesando: Trump ha sido devorado por el caos que él puso en march...
    -> Texto completo (7771 chars)
  Procesando: "Soy ciudadano de EEUU e Israel ha destruido mi casa c

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [10]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  Estrasburgo rechaza paralizar la eutanasia de Noelia
Origen:  completo
Chars:   3372
Preview: Barcelona
El Tribunal Europeo de Derechos Humanos (TEDH) ha rechazado las cautelares que solicitó el padre de Noelia para paralizar la eutanasia de la joven. El progenitor, representado por Abogados C...

Artículo 2
Fuente:  ABC
Título:  Las rebajas fiscales del Gobierno ante la guerra de Irán supondrán un ahorro de 191 millones en Castilla y León
Origen:  completo
Chars:   1457
Preview: Las rebajas fiscales del Gobierno ante la guerra de Irán supondrán un ahorro de 191 millones en Castilla y León
Carriedo destaca que «el mayor esfuerzo de recaudación corresponde a las autonomías» y v...

Artículo 3
Fuente:  ABC
Título:  La receta electrónica llegará a partir del 7 de abril a más de 68.600 mutualistas en Castilla y León
Origen:  completo
Chars:   4477
Preview: La receta electrónica llegará a partir del 7 de abril a más de 68.600 mutualistas en Castilla y León


## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [12]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 27 ARTÍCULOS

-> Agrupando artículos por temas...
   5 temas identificados:
   - Conflicto en Oriente Medio y Política Exterior de EEUU (7 fuentes: ElDiario, ElDiario, ElDiario, Europa Press, La Vanguardia, La Vanguardia, Antena 3)
   - Elecciones y Cambios de Gobierno en Andalucía (4 fuentes: Europa Press, Europa Press, RTVE, El Pais)
   - Política de Vivienda y Alquiler en España (4 fuentes: 20minutos, 20minutos, 20minutos, El Mundo)
   - Controversias y Finanzas de Vox (2 fuentes: El Pais, El Pais)
   - Eutanasia (1 fuentes: ABC)

-> Generando resúmenes por tema...
   [1/5] Conflicto en Oriente Medio y Política Exterior de EEUU (7 fuentes)...
      OK (1061 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [2/5] Elecciones y Cambios de Gobierno en Andalucía (4 fuentes)...
      OK (856 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [3/5] Política de Vivienda y Alquiler en España (4 fuentes)...
      OK (847 chars)
  

## Paso 3: Síntesis de voz (T2S)

In [13]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       5184
  Audio:       audios/audio_2026-03-24_09-25-25.mp3
Audio generado! (1.78 MB)
  Generando subtítulos con Whisper...


c:\Users\flore\anaconda3\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Subtítulos generados: audios/subtitulos_2026-03-24_09-25-25.vtt


In [14]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 12470 chars
WEBVTT

00:00:00.000 --> 00:00:01.219
Muy buenas tardes.

00:00:02.259 --> 00:00:03.379
Les damos la bienvenida

00:00:03.379 --> 00:00:04.780
a nuestro boletín informativo,

00:00:05.259 --> 00:00:06.960
donde repasaremos los acontecimientos

00:00:06.960 --> 00:00:08.199
más destacados del día,

0


## Paso 4: Generación del video resumen 

In [15]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [16]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')

ruta_video = generar_video(ruta_audio, ruta_subtitulos, resumenes, PEXELS_API_KEY)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    311.3s
   Temas:             5
   Duración por tema: 62.3s

2. Buscando imágenes en Pexels...
  Tema: Conflicto en Oriente Medio y Política Exterior de ...
    Keywords: 'conflicto oriente medio política'
    Imagen guardada: imagenes/img_609253369764424839.jpg
  Tema: Elecciones y Cambios de Gobierno en Andalucía...
    Keywords: 'elecciones cambios gobierno andalucía'
    Imagen guardada: imagenes/img_2042950111236359024.jpg
  Tema: Política de Vivienda y Alquiler en España...
    Keywords: 'política vivienda alquiler españa'
    Imagen guardada: imagenes/img_5889906250981051858.jpg
  Tema: Controversias y Finanzas de Vox...
    Keywords: 'controversias finanzas'
    Imagen guardada: imagenes/img_6637597530826912901.jpg
  Tema: Eutanasia...
    Keywords: 'eutanasia'
    Imagen guardada: imagenes/img_6590453146620290207.jpg

3. Creando clips por tema...
   [1/5] Conflicto en Oriente Medio y Política Ex...
   [2/5] Elecciones